In [2]:
# Importing necessary libraries
import numpy as np 
import pandas as pd 
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.datasets as datasets
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.utils as utils
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from tqdm import tqdm

torch.manual_seed(0)

In [39]:
# Transform for Data Augmentation
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229, 0.224, 0.225]) 
])

train_dataset = datasets.ImageFolder('/kaggle/input/dlp-ga-9-image-classification/train', transform=transform)
test_dataset = datasets.ImageFolder('/kaggle/input/dlp-ga-9-image-classification', transform=transform)

test_indices = [i for i, label in enumerate(test_dataset.targets) if label == 0]
test_dataset = Subset(test_dataset, test_indices)

In [10]:
# New Code
# test_dataset_new = datasets.ImageFolder('/kaggle/input/testing-2', transform=transform)
# test_dataset_new

Dataset ImageFolder
    Number of datapoints: 98
    Root location: /kaggle/input/testing-2
    StandardTransform
Transform: Compose(
               Resize(size=256, interpolation=bilinear, max_size=None, antialias=True)
               CenterCrop(size=(224, 224))
               ToTensor()
               Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
           )

In [14]:
# Creating a Validation Set
training_size_frac = 0.8
train_size = int(training_size_frac*len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = random_split(train_dataset, [train_size,val_size])

In [15]:
# Defining the label map
label_dict = {
    0: "Apple_Apple_scab",
    1: "Apple_Black_rot",
    2: "Apple_Cedar_apple_rust",
    3: "Apple_healthy",
    4: "Blueberry_healthy",
    5: "Cherry_including_sour_Powdery_mildew",
    6: "Cherry_including_sour_healthy",
    7: "Corn_maize_Cercospora_leaf_spot_Gray_leaf_spot",
    8: "Corn_maize_Common_rust_",
    9: "Corn_maize_Northern_Leaf_Blight",
    10: "Corn_maize_healthy",
    11: "Grape_Black_rot",
    12: "Grape_Esca_Black_Measles_",
    13: "Grape_Leaf_blight_Isariopsis_Leaf_Spot_",
    14: "Grape_healthy",
    15: "Orange_Haunglongbing_Citrus_greening_",
    16: "Peach_Bacterial_spot",
    17: "Peach_healthy",
    18: "Pepper,_bell_Bacterial_spot",
    19: "Pepper,_bell_healthy",
    20: "Potato_Early_blight",
    21: "Potato_Late_blight",
    22: "Potato_healthy",
    23: "Raspberry_healthy",
    24: "Soybean_healthy",
    25: "Squash_Powdery_mildew",
    26: "Strawberry_Leaf_scorch",
    27: "Strawberry_healthy",
    28: "Tomato_Bacterial_spot",
    29: "Tomato_Early_blight",
    30: "Tomato_Late_blight",
    31: "Tomato_Leaf_Mold",
    32: "Tomato_Septoria_leaf_spot",
    33: "Tomato_Spider_mites_Two-spotted_spider_mite",
    34: "Tomato_Target_Spot",
    35: "Tomato_Tomato_Yellow_Leaf_Curl_Virus",
    36: "Tomato_Tomato_mosaic_virus",
    37: "Tomato_healthy"
}

In [16]:
# Defining the DataLoaders for the train, val and test splits
batch_size=128
train_loader = DataLoader(dataset=train_dataset,batch_size=batch_size,shuffle=True)
val_loader = DataLoader(dataset=val_dataset,batch_size=batch_size,shuffle=False)
test_loader = DataLoader(dataset=test_dataset,batch_size=batch_size,shuffle=False)

print(len(train_loader.dataset))
print(len(val_loader.dataset))
print(len(test_loader.dataset))

34743
8686
10876


In [3]:
# Ensuring correct device before starting training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Model will be trained using {device}")

Model will be trained using cuda


In [23]:
# Defining calculate correct output prediction function
def calculate_correct_predictions(out, labels):
    _, pred = torch.max(out, dim=1)
    return torch.sum(pred==labels).item()

In [5]:
# Initializing the model
model = models.vit_b_16(weights=True)
for param in model.parameters():
    param.requires_grad=False
model.heads = nn.Sequential(
    nn.Linear(in_features=768, out_features=38, bias=True)
)
model = model.to(device)
model

VisionTransformer(
  (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  (encoder): Encoder(
    (dropout): Dropout(p=0.0, inplace=False)
    (layers): Sequential(
      (encoder_layer_0): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (dropout): Dropout(p=0.0, inplace=False)
        (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): MLPBlock(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.0, inplace=False)
          (3): Linear(in_features=3072, out_features=768, bias=True)
          (4): Dropout(p=0.0, inplace=False)
        )
      )
      (encoder_layer_1): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_a

In [54]:
# Defining the train_model function
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs, device):
    # Iterating through each epoch
    for epoch in range(1, num_epochs+1):
        print(f"------------------------ Epoch No. {epoch} ------------------------")
        running_loss = 0
        correct_train, correct_val = 0, 0
        
        # tqdm library is used for displaying the progress bar
        with tqdm(total=len(train_loader), desc=f'Epoch {epoch}/{num_epochs}', unit='batch') as tepoch:
            # Iterating through each batch
            for images, labels in train_loader:
                images = images.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()
                
                # Output is obtained
                outputs = model(images)
                
                # Loss is calculated and propagated backwards
                loss = criterion(outputs, labels)
                loss.backward()
                running_loss += loss.item()
                
                # Gradients are updated
                optimizer.step()
                
                # No. of correct predictions are calculated
                correct_train += calculate_correct_predictions(outputs, labels)

                tepoch.set_postfix(loss=running_loss / (tepoch.n + 1), accuracy=(correct_train / ((tepoch.n + 1) * train_loader.batch_size)) * 100)  # Update progress bar with current loss and accuracy
                tepoch.update()  # Increment the progress bar

        # Calculating training accuracy score for the epoch
        acc_train = (correct_train / len(train_loader.dataset)) * 100
        print(f"\nEpoch {epoch}: Accuracy on Train Set = {acc_train:.2f} %, Training Cost = {running_loss:.2f}")
        
        # Performance on validation dataset is checked
        print(f"Validation begins for Epoch {epoch}...")
        
        # 'torch.no_grad()' is used to disable gradient calculation during the inference or testing phase
        with torch.no_grad():
            # Iterating through each batch
            for x_val, y_val in val_loader:
                x_val = x_val.to(device)
                y_val = y_val.to(device)
                
                # Output is obtained
                z = model(x_val)
                # Correct no. of predictions is calculated
                correct_val += calculate_correct_predictions(z, y_val)
        
        # Calculating validation accuracy score for the epoch
        acc_val = (correct_val / len(val_loader.dataset)) * 100
        print(f"Epoch {epoch}: Accuracy on Validation Set = {acc_val:.2f} %")
    
    return model


In [55]:
# Used for freeing up the memory
torch.cuda.empty_cache()

In [56]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

num_epochs = 20

model = train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs, device)

------------------------ Epoch No. 1 ------------------------


Epoch 1/20: 100%|██████████| 272/272 [04:54<00:00,  1.08s/batch, accuracy=85.5, loss=0.678]



Epoch 1: Accuracy on Train Set = 85.64 %, Training Cost = 184.45
Validation begins for Epoch 1...
Epoch 1: Accuracy on Validation Set = 94.84 %
------------------------ Epoch No. 2 ------------------------


Epoch 2/20: 100%|██████████| 272/272 [04:45<00:00,  1.05s/batch, accuracy=95.7, loss=0.215]



Epoch 2: Accuracy on Train Set = 95.87 %, Training Cost = 58.35
Validation begins for Epoch 2...
Epoch 2: Accuracy on Validation Set = 96.42 %
------------------------ Epoch No. 3 ------------------------


...


Epoch 18: Accuracy on Train Set = 99.20 %, Training Cost = 12.83
Validation begins for Epoch 18...
Epoch 18: Accuracy on Validation Set = 98.27 %
------------------------ Epoch No. 19 ------------------------


Epoch 19/20: 100%|██████████| 272/272 [04:38<00:00,  1.03s/batch, accuracy=99, loss=0.0456]  



Epoch 19: Accuracy on Train Set = 99.25 %, Training Cost = 12.40
Validation begins for Epoch 19...
Epoch 19: Accuracy on Validation Set = 98.33 %
------------------------ Epoch No. 20 ------------------------


Epoch 20/20: 100%|██████████| 272/272 [04:49<00:00,  1.07s/batch, accuracy=99.1, loss=0.044] 



Epoch 20: Accuracy on Train Set = 99.26 %, Training Cost = 11.98
Validation begins for Epoch 20...
Epoch 20: Accuracy on Validation Set = 98.38 %


In [57]:
# Storing Test Set Predictions
def get_predictions(model, test_loader, device, label_map):
    model.eval()  # Set the model to evaluation mode
    pred_labels = []
    # Gradient calculation is disabled
    with torch.no_grad():
        # Iterating through each batch of test dataloader
        for x_test, y_test in test_loader:
            x_test = x_test.to(device)
            y_test = y_test.to(device)
            
            # Output is obtained
            z = model(x_test)
            
            # Predicted labels are collected
            _, predicted = torch.max(z.data, 1)
            
            # Current batch's labels and predictions are stored
            pred_labels.extend(predicted.cpu().numpy())
        
    # Accuracy is calculated
    pred_labels = np.array(pred_labels)
    return pred_labels


In [17]:
## New Code
# test_loader_new = DataLoader(dataset=test_dataset_new,batch_size=batch_size,shuffle=False)

In [38]:
## New code
# filenames = [sample.split("/")[-1] for sample,_ in test_loader_new.dataset.samples]
# filenames

['Potato_Early_Blight Pasted image (2).png',
 'Potato_Early_Blight Pasted image (3).png',
 'Potato_Early_Blight Pasted image.png',
 'Potato_Early_Blight images (100).jpg',
 'Potato_Early_Blight images (93).jpg',
 'Potato_Early_Blight images (95).jpg',
 'Potato_Early_Blight images (97).jpg',
 'Potato_Early_Blight images (98).jpg',
 'Potato_Early_Blight images (99).jpg',
 'Potato_Healthy images (100).jpg',
 'Potato_Healthy images (95).jpg',
 'Potato_Healthy images (96).jpg',
 'Potato_Healthy images (98).jpg',
 'Potato_Healthy images (99).jpg',
 'Potato_Healthy images.jpg',
 'Potato_Late_Blight OIP.z-s6xYuTB_0t2jPnLmQZ9QHaFW.jpg',
 'Potato_Late_Blight OIP.zJWFM0H2-8al4g4fq2cVHgHaE8.jpg',
 'Potato_Late_Blight OIP.zMFPLyA4jIs-NOwXlVnkjwHaFc.jpg',
 'Potato_Late_Blight OIP.zXWE-4Dc4UU5iHbCa1kl5QHaE_.jpg',
 'Potato_Late_Blight OIP.zpJpTzQ5xgXE2ktsHFGkLwHaE8.jpg',
 'Potato_Late_Blight OIP.zqKfE-ahKSJNtC4mKEhK_QHaEe.jpg',
 'Potato_Late_Blight Phytophthora-infestans-a-major-cause-of-late-blight-d

In [60]:
# Creating the CSV file for the submission [For Kaggle or comparing the results]
# predictions = get_predictions(model, test_loader, device, label_dict)
# parent_dataset = test_loader.dataset.dataset
# filenames = [parent_dataset.samples[i][0].split("/")[-1].split(".")[0] for i in test_indices]

# submission_df = pd.DataFrame({"Image_ID":filenames, "Label": predictions})
# submission_df.to_csv("/kaggle/working/submission.csv", index=False)